# Payload Builder

Esta é uma documentação detalhada e didática da classe PayloadBuilder, projetada para padronizar a telemetria e o logging de aplicações Python.

## Visão Geral

A classe `PayloadBuilder` atua como uma **fábrica de mensagens e estruturas de dados**. Seu objetivo principal é garantir que, independentemente de onde um log seja gerado no sistema, ele possua uma estrutura consistente.

Ela resolve o problema comum de logs desorganizados, permitindo que o desenvolvedor gere tanto uma **string amigável para leitura humana** (console) quanto um **objeto JSON estruturado** para bancos de dados NoSQL (como MongoDB), mantendo o rastreio através de IDs e metadados.

---

## Fluxo de Execução

O fluxo típico de utilização da classe segue estes passos:

1. **Instanciação**: O objeto é criado, geralmente recebendo o contexto global (ID do log, nome do arquivo ou flags de categoria).
2. **Processamento de Metadados**: Ao registrar um evento, a classe avalia se existem dados extras (`metadata`) e como eles devem ser exibidos (se formatados como JSON identado ou string simples).
3. **Composição da Mensagem**: O método `build_message` concatena as partes (função, mensagem, contexto) usando um delimitador (`|`).
4. **Estruturação para Persistência**: Caso o log precise ser salvo em banco, o método `build_mongo_payload` organiza as mesmas informações em um dicionário Python com timestamps precisos.

---

## Resumo dos Métodos

| **Método** | **Responsabilidade** |
| --- | --- |
| `__init__` | Configura o contexto do builder (ID, arquivo, flags e preferências de formato). |
| `format_metadata_payload` | Trata a visualização de dicionários extras, convertendo-os em strings ou JSON identado. |
| `build_message` | Gera a string final formatada para exibição em logs de console ou arquivos de texto. |
| `build_mongo_payload` | Cria um dicionário estruturado com data/hora para inserção em bancos de dados. |

---

## Arquitetura e Insights

- **Separação de Preocupações**: A classe separa a *lógica de formatação* da *lógica de transporte* (envio para o banco). Ela não "loga" nada, ela apenas "constrói o payload".
- **Rastreabilidade (Observabilidade)**: O uso do `log_id` permite correlacionar diferentes entradas de log que pertencem à mesma requisição ou processo.
- **Resiliência**: O método de formatação possui um bloco `try/except` para garantir que, mesmo se a serialização JSON falhar, o log não quebre a aplicação, fazendo fallback para uma string simples.

---

## Detalhamento da Classe

## Classe `PayloadBuilder`

**Descrição**

Responsável por centralizar a composição de mensagens de log, tratamento de metadados e organização de informações contextuais para garantir padronização em diferentes saídas (console e banco de dados).

**Argumentos**

- `log_id` (str, opcional): Identificador único da transação ou processo.
- `flag` (str, opcional): Tag para categorização (ex: "SISTEMA_PAGAMENTO").
- `file_name` (str, opcional): Nome do arquivo Python onde o log ocorreu.
- `format_metadata` (bool): Se `True`, formata dicionários de metadados com identação JSON.

---

## Métodos

## 1. format_metadata_payload()

**Descrição**

Formata o dicionário de metadados para uma representação em string.

**Argumentos**

- `metadata` (Dict[str, Any]): Dados adicionais para incluir no log.
- `show_metadata` (bool): Flag que autoriza ou não a inclusão dos dados na saída.

**Retornos**

- `Optional[str]`: String formatada ou `None` se os dados não devem ser exibidos.

**Raises**

- **Nenhum**: Captura exceções internamente e retorna a representação em string bruta (`str(metadata)`) em caso de erro.

**Exemplos**

```bash
# Com format_metadata=True na classe
builder.format_metadata_payload({"status": 200}, True)
# Retorna: "\n{\n    "status": 200\n}\n"`
```

---

## 2. build_message()

**Descrição**

Monta a string de log final, unindo os componentes por um separador vertical (`|`).

**Argumentos**

- `func_name` (str): Nome da função de origem.
- `message` (str): Texto explicativo do log.
- `metadata` (Dict): Dados extras.
- `show_metadata` (bool): Se os dados extras devem aparecer nesta mensagem.

**Retornos**

- `str`: Linha de log completa e formatada.

**Raises**

- `ValueError`: Se nenhum parâmetro for fornecido (mensagem vazia).

**Exemplos**

```bash
builder.build_message("process_data", "Sucesso", {"items": 5}, True)
# Retorna: 'process_data() | Sucesso | metadata={'items': 5} | log_id=...'
```

---

## 3. build_mongo_payload()

**Descrição**

Prepara um objeto pronto para ser inserido em coleções do MongoDB ou bancos similares.

**Argumentos**

- `level` (str): Nível do log (DEBUG, INFO, ERROR, etc).
- `func_name` (str): Função de origem.
- `message` (str): Conteúdo do log.
- `metadata` (Dict): Dados estruturados.

**Retornos**

- `Dict[str, Any]`: Dicionário contendo campos padronizados e timestamp.

**Raises**

- `ValueError`: Se o nível (level) não for informado.

**Exemplos**

```bash
builder.build_mongo_payload("error", "save_user", "Falha de conexão", {"db": "prod"})
# Retorna: {'log_id': '...', 'level': 'ERROR', 'time': '2026-03-18...', ...}
```